## Oltre i semplici “clic”: la shell come strumento essenziale per l’analisi dei dati e il calcolo avanzato

Quando si affrontano attività di analisi o calcolo più complesse, affidarsi esclusivamente alle interfacce grafiche non è sufficiente. Diventa quindi importante saper lavorare tramite comandi da shell e script. Questi strumenti sono utili in numerosi contesti: gestione del file system, operazioni di basso livello, configurazioni locali e remote avanzate, accesso a risorse esterne e relativa amministrazione.

Esistono diversi linguaggi di shell, generalmente abbastanza simili tra loro. In questa lezione ci concentreremo su BASH, interprete predefinito in molti sistemi GNU/Linux. Su macOS l’applicazione di riferimento è `Terminale`: fino a Catalina utilizzava BASH come shell predefinita, mentre successivamente è passata a ZSH, che presenta comunque molte analogie.

Anche Windows dispone di strumenti equivalenti, che però non verranno approfonditi in questo notebook.


## Programmazione di script con la shell Bash

### Bash

Il nome Bash deriva dall’acronimo “Bourne-Again SHell”. Come descritto nella relativa [pagina Wikipedia](https://en.wikipedia.org/wiki/Bash_(Unix_shell)), Bash è un interprete di comandi che viene normalmente eseguito in una finestra testuale. L’utente può inserire istruzioni che avviano determinate operazioni, oppure raccogliere più comandi in un file, chiamato *shell script*, ed eseguirli in sequenza. Come molte shell Unix, supporta wildcard nei nomi dei file, pipe, *here document*, sostituzione dei comandi, variabili e strutture di controllo per condizioni e iterazioni.

### Shell

Una shell può essere considerata un elaboratore di comandi che consente di eseguire istruzioni sia in modalità interattiva sia in modalità non interattiva.

### Scripting

Lo scripting permette di automatizzare una serie di comandi che, altrimenti, dovrebbero essere inseriti manualmente uno alla volta.


---

Nelle sezioni successive esamineremo le funzionalità fondamentali di Bash, limitandoci a una prima introduzione a uno strumento estremamente ricco. Come accade per Python, sul web è disponibile una grande quantità di documentazione utile per quasi ogni operazione. Inoltre, per la maggior parte dei comandi è possibile consultare il manuale tramite `man`; molti programmi mettono a disposizione anche l’opzione `--help`.

Jupyter consente di eseguire comandi e script Bash direttamente all’interno di un notebook Python, oppure utilizzando un emulatore di terminale dedicato.

Gli esempi riportati di seguito sono comunque pensati soprattutto per essere copiati e provati nella shell del proprio computer. Gli script, invece, possono essere scritti e modificati in file separati tramite l’[editor di testo](https://www.javatpoint.com/linux-text-editors) preferito.


## Orientarsi ed esplorare il file system


```bash

# Questo è un commento

# Chi sono? In altre parole, quale account sto utilizzando?
whoami

# Dispongo dei privilegi di superutente?
sudo -l

# Qual è il nome del computer?
hostname

# In quale directory mi trovo?
pwd
# La directory corrente è indicata con ".", mentre ".." rappresenta quella superiore.
# pwd restituisce il contenuto della variabile globale $PWD, che vedremo più avanti.

# Spostarsi in una determinata directory
# Salire alla directory superiore
cd .. 
# Tornare alla directory visitata in precedenza
cd -  
# Spostarsi nella propria directory home
cd $HOME

# Creare una nuova directory
mkdir test
cd test
mkdir -p tmp/foo 
# Eliminare la directory appena creata
rm -r tmp/foo # L’opzione "-r" è necessaria quando si eliminano directory

# Controllare lo spazio occupato dai dati nella propria home
du -h $HOME
# Verificare l’utilizzo dello spazio nei principali file system
df -h

# Visualizzare il contenuto di una directory
touch tmp_file # Creiamo un file di prova, che verrà utilizzato più avanti
ls -latrh 
# Opzioni: -l -> formato elenco, -a -> include gli elementi nascosti,
# -t -> ordina per data, -r -> inverte l’ordine, -h -> mostra dimensioni leggibili
```


### Come leggere l’output di `ls -l`

L’output è composto da sette campi:

1. Permessi
2. Numero di hard link
3. Proprietario del file
4. Gruppo associato al file
5. Dimensione del file
6. Data e ora dell’ultima modifica
7. Nome del file

Il primo campo può essere interpretato nel modo seguente:

`Tipo di file - Permessi del proprietario - Permessi del gruppo - Permessi degli altri utenti`

I simboli relativi ai permessi hanno questo significato:

- `r` = autorizzazione alla lettura
- `w` = autorizzazione alla scrittura
- `x` = autorizzazione all’esecuzione
- `-` = autorizzazione assente


## Gestione dei file

Per approfondire la sintassi utilizzata nella modifica dei permessi, si può consultare la relativa [pagina Wikipedia](https://en.wikipedia.org/wiki/Chmod).


``` bash
# Scrivere del testo in un file
echo "Inserisci del testo in un file" > tmp_file 
more tmp_file 

# Il simbolo ">" sostituisce il contenuto eventualmente già presente
echo "Inserisci un altro testo nel file" > tmp_file 
more tmp_file 

# Il simbolo ">>" aggiunge il testo in fondo al contenuto esistente
echo "Aggiungi del testo al file" >> tmp_file 
more tmp_file 

# Modificare i permessi: in questo caso si permette agli "altri" utenti
# di leggere e scrivere il file
ls -l tmp_file
chmod o+rw tmp_file
ls -l tmp_file

# Copiare un file; naturalmente è possibile specificare anche un percorso assoluto
cp tmp_file ./tmp_file_copy

# Eliminare un file
rm tmp_file_copy

# Cercare un file
# Sintassi generale: find /percorso/della/directory -name "nomefile"
cd ../
find . -name "tmp_file"
find . -name "tmp*"
cd -

```   


## Espressioni regolari (RegExp)

Le espressioni regolari sono uno strumento molto potente per manipolare stringhe, cercare testo, effettuare sostituzioni e svolgere operazioni simili. La loro sintassi può risultare inizialmente poco intuitiva e piuttosto articolata, ma viene impiegata in numerosi linguaggi di programmazione e vale quindi la pena impararla. In questo ambito è spesso indispensabile consultare regolarmente i manuali.

Negli esempi successivi utilizzeremo le espressioni regolari insieme ai comandi `grep` e `sed`.

Copiare il contenuto seguente in un file chiamato `data.csv`.


``` bash
# Estrarre la riga che contiene la parola "sensore"
grep "sensore" data.csv

# Estrarre i metadati
grep "^#" data.csv 

# Ottenere i dati effettivi, cioè le righe che non iniziano con "#"
grep -v "^#" data.csv 

# Contare il numero di righe contenenti i dati
grep -c -v "^#" data.csv 

# Cercare il sensore tramite il suo nome
grep "X[a-z].v*" data.csv 
# Le ricerche distinguono tra lettere maiuscole e minuscole
grep "X[A-Z].v*" data.csv 

# Usare cut per estrarre una parte della riga, in questo caso l’orario iniziale
# -d " " significa: separa la riga usando lo spazio (d=delimitatore)
# (Questo comando prende la riga che contiene la parola “iniziata” e ne estrae l’ottavo campo)
grep "iniziata" data.csv | cut -f8 -d " "

# Sostituire ottobre con novembre
sed -e "s/ottobre/novembre/" data.csv

# Aggiungere ".0" a tutti i numeri formati da una sola cifra
# \{1\} indica esattamente una cifra
sed -e "s/\b[0-9]\{1\}\b/&.0/" data.csv 

```


## Variabili

Le variabili possono essere locali oppure globali. Le variabili locali vengono definite con la consueta assegnazione diretta, mentre per rendere globale una variabile occorre utilizzare il comando `export`. Per accedere al valore di una variabile si antepone il simbolo `$` al suo nome.

Le variabili globali sono ampiamente utilizzate da applicazioni e componenti del sistema sui quali spesso non si desidera intervenire direttamente. È quindi opportuno modificarle soltanto quando si conoscono bene le conseguenze. Per convenzione, i loro nomi sono generalmente scritti in lettere maiuscole; per le proprie variabili è invece preferibile utilizzare nomi in minuscolo.

Il comando `env` mostra l’elenco delle variabili globali attualmente definite.

Un esempio importante è `PATH`, che contiene l’elenco delle directory in cui vengono cercati i programmi eseguibili che possono essere avviati senza indicarne il percorso assoluto.


``` bash
# Visualizzare l’elenco completo delle variabili globali
env

# Qual è il contenuto della variabile PATH?
echo $PATH

# Definire una propria variabile globale
export MY_GLOBAL_VARIABLE=pippo
echo $MY_GLOBAL_VARIABLE

# Definire una variabile locale
variable=7
echo $variable

```


## Output standard ed errore standard

Quando si esegue un comando possono verificarsi tre situazioni principali: il comando produce l’output previsto, genera un errore oppure termina senza mostrare alcun risultato.

Spesso, ad esempio per creare file di log, è necessario salvare separatamente i diversi tipi di output. La notazione `>` permette di reindirizzare lo standard output (*stdout*) verso un file; `2>` reindirizza invece lo standard error (*stderr*), mentre `&>` invia nello stesso file sia l’output standard sia gli errori.


```bash

# Esempi di standard error
ls -l pippo
ls -l pippo 2> err.txt
more err.txt 
ls -l pippo > err.txt
more err.txt 

# Esempio di standard output
ls -l err.txt > log.txt
more log.txt 

# Reindirizzare sia output sia errori
ls -l pippo err.txt &> log.txt
more log.txt 

```


## Confronto tra numeri e stringhe e operazioni numeriche

La tabella seguente riassume gli operatori utilizzati per confrontare valori numerici e stringhe.

| Descrizione | Confronto numerico | Confronto tra stringhe |
| ----------- | ----------- | ----------- |
| minore di | -lt | < |
| maggiore di | -gt | > |
| uguale | -e | = |
| diverso | -ne | != |
| minore o uguale | -le | N.D. |
| maggiore o uguale | -ge | N.D. |

Una particolarità è che il risultato di un confronto vale `1` quando la condizione è falsa e `0` quando è vera.

Gli esempi seguenti mostrano come accedere direttamente al codice restituito dal confronto. Questo meccanismo viene usato soprattutto nelle istruzioni condizionali.


``` bash
# Assegnazione delle variabili
a=1
b=2

# Eseguire il confronto; si notino le parentesi quadre
[ $a -lt $b ]
# Controllare il risultato dell’operazione precedente
echo $?

[ $a -gt $b ]; echo $?
[ $a -ne $b ]; echo $?
```


Le operazioni aritmetiche possono essere eseguite in diversi modi. Una soluzione molto comune è l’*Arithmetic Expansion*, apprezzata per la sua semplicità, anche se consente un insieme più limitato di operazioni. Esistono inoltre strumenti come `expr` e `let`; qui verranno mostrati anche alcuni esempi con `bc`, che offre una sintassi intuitiva e funzionalità più ampie.


```bash
# Espansioni aritmetiche
echo $(( 10*5 + 15 ))
echo $(( $a + $b*3 ))

# Utilizzo di bc
echo '8.5 / 2.3' | bc
sqr=$( echo 'scale=6;sqrt(2)' | bc)
echo $sqr

```


## Istruzioni condizionali

Come negli altri linguaggi di programmazione, anche in Bash le istruzioni condizionali hanno un ruolo fondamentale e vengono impiegate molto frequentemente. Sono particolarmente adatte all’interno degli script, che vedremo in seguito, ma possono essere utilizzate anche direttamente dalla riga di comando. La sintassi è simile a quella adottata da molti altri linguaggi.


```bash
a=400
b=200

# Comando scritto su una sola riga
if [ $a -gt $b ]; then echo "$a è maggiore di $b! "; fi

# Il blocco seguente deve essere inserito in uno script
if [ $a -lt $b ]; then
    echo "$a è minore di $b! "
else
    echo "$a è maggiore di $b! "
fi
```


## Cicli

Per i cicli valgono considerazioni simili a quelle fatte per le istruzioni condizionali. Esistono varie modalità per implementarli; di seguito ne verranno presentate alcune.


```bash
# Alcuni esempi di sintassi per i cicli for
for i in 1 2 3; do echo $i; done

for i in {1..10}; do echo $i; done

for i in $(seq 1 2 20); do echo $i; done

for (( i=1; i<=5; i++ )); do echo $i; done

for i in `ls .`; do echo $i; done 

for file in ./*; do if [ "${file}" == "./log.txt" ]; then break; fi; echo $file; done

# Sono disponibili anche i cicli while e until
counter=0                                                                               
while [ $counter -lt 3 ]; do let counter+=1; echo $counter; done

```


## Creazione di script

Tutte le istruzioni viste finora, insieme a molte altre, possono essere combinate all’interno di uno script. Di seguito è riportato un esempio. 



```bash
#!/bin/bash

# Verificare che l’utente abbia fornito un argomento in ingresso (per esempio: ./my_script.sh output.txt) 
if [ -z $1 ]
then
    echo "questo script richiede come input il nome del file da creare"
    exit
fi

# Controllare se il file esiste
if [ ! -f "./$1" ]
then
    echo "il file ./$1 non esiste! Verrà utilizzato un valore predefinito"
    file="newfile.txt"
else
    file=./$1
fi

touch $file

for (( i=1; i<=5; i++ ))
do
    echo "aggiungi la riga $i" >> $file
done
```


L’istruzione iniziale `#!/bin/bash` indica alla shell quale interprete deve essere utilizzato per eseguire i comandi successivi.

È possibile salvare queste righe in un nuovo file, ad esempio `my_script.sh`, e provare a eseguirlo. Prima dell’esecuzione è necessario renderlo eseguibile.


```bash
chmod +x my_script.sh

./my_script.sh output.txt
```


## Documentazione

Come già ricordato, è disponibile molta documentazione sia direttamente tramite `man` sia online. Un altro comando particolarmente utile è `history`, che mostra l’elenco dei comandi eseguiti nella shell corrente. Dopo una lunga sessione di lavoro da terminale, può essere conveniente eseguire questo comando e reindirizzarne lo standard output verso un file di log, in modo da poter consultare successivamente la cronologia delle operazioni.




# Esercizio 

### 1.a

Creare nella propria directory home una nuova cartella chiamata `students`.

Scaricare da https://www.dropbox.com/scl/fi/bxv17nrbrl83vw6qrkiu9/LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7 il file CSV contenente l’elenco degli studenti del laboratorio, utilizzando il comando `wget`, e copiarlo nella cartella `students`.

Prima di effettuare il download, verificare che il file non sia già presente.

### 1.b

Creare due nuovi file (usando `grep` e `>`):

* uno contenente gli studenti appartenenti al corso PoD;
* uno contenente gli studenti appartenenti al corso di Physics.

### 1.c

Per ogni lettera dell’alfabeto, contare quanti studenti hanno un cognome che inizia con quella lettera.
`-n -2 --> elimina la prima riga (intestazione)` <br>
`-d',' usa come separatore la virgola e f1 seleziona il primo campo (cognome)`<br>
`-i ignora la distinzione tra maiuscole e minuscole;`<br>
`-c non stampa le righe, ma le conta;`<br>
`^ indica l’inizio della riga;` <br>
comando:
`numero=$(tail -n +2 studenti_Physics.csv | cut -d',' -f1 | grep -ic "^$lettera")`

### 1.d

Individuare la lettera a cui corrisponde il maggior numero di studenti.
`done | sort -k2 -nr | head -n 1`

- k2: la seconda colonna, cioè il numero degli studenti;
- n: ordinamento numerico;
- r: ordine inverso, quindi dal valore maggiore al minore.

### 1.e

Supporre che gli studenti siano numerati in base alla loro posizione nel file: il primo studente ha numero 1, il secondo numero 2 e così via.

Dividere gli studenti in gruppi secondo il resto della divisione per 18. I gruppi saranno quindi formati nel modo seguente:

* gruppo 1: studenti 1, 19, 37, ecc.;
* gruppo 2: studenti 2, 20, 38, ecc.;
* e così via.

Salvare ogni gruppo in un file separato.

`awk 'NR>1 { gruppo=((NR-2)%18)+1; print > ("gruppo_" gruppo ".csv") }' studenti_Physics.csv` <br>

The awk command `awk 'condizione { azioni }' file` has options to change how it works:

    -F - Set what separates the data fields
    -v - Set a variable to be used in the script
    -f - Use a file as the source of the awk program


